In [1]:
import sys
sys.path.append('..')

from src.parser import parse_bgl_fields, build_miner
from src.features import build_windows, attach_window_labels

In [2]:
df = parse_bgl_fields('../data/raw/BGL/BGL_2k.log')
df.head()

Parsed: 2000 | Skipped: 0


,label,timestamp,date,node,time,noderepeat,type,component,level,content
0,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.675872,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected
1,-,1117838573,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.53.276129,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected
2,-,1117838976,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.49.36.156884,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected
3,-,1117838978,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.49.38.026704,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected
4,-,1117842440,2005.06.03,R23-M0-NE-C:J05-U01,2005-06-03-16.47.20.730545,R23-M0-NE-C:J05-U01,RAS,KERNEL,INFO,63543 double-hummer alignment exceptions


In [3]:
miner = build_miner('../drain3_bgl.ini', '../models/bgl_drain_state_sample.bin')
df['EventId'] = [miner.add_log_message(str(c))['cluster_id'] for c in df['content']]

print("Distinct templates mined:", df['EventId'].nunique())

Distinct templates mined: 105


In [4]:
for cluster in list(miner.drain.clusters)[:10]:
    print(cluster.cluster_id, '|', cluster.get_template())

1 | instruction cache parity error corrected
2 | <NUM> double-hummer alignment exceptions
3 | CE sym <NUM>, at <HEX>, mask <HEX>
4 | ciod: failed to read message prefix on control stream (CioStream socket to <NUM>.<NUM>.<NUM>.<NUM>:<NUM>
5 | generating core.<NUM>
6 | force load<PATH> alignment...............<NUM>
7 | ciod: cpu <NUM> at treeaddr <NUM> sent unrecognized message <HEX>
8 | ciod: LOGIN <*> failed: No such file or directory
9 | <NUM> ddr errors(s) detected and corrected on rank <NUM>, symbol <NUM>, bit <NUM>
10 | data TLB error interrupt


In [5]:
windows = build_windows(df, window_size=100)
labeled = attach_window_labels(windows, df, window_size=100)

print("Windows:", len(labeled))
print("Anomalous windows:", int(labeled['y'].sum()), "/", len(labeled),
      f"({labeled['y'].mean() * 100:.1f}%)")
labeled.head()

Windows: 20
Anomalous windows: 12 / 20 (60.0%)


,WindowId,EventSequence,y,causes
0,0,"[1, 1, 1, 1, 2, 2, 2, 3, 4, 4, 3, 5, 5, 5, 5, ...",1,[APPREAD]
1,1,"[3, 5, 5, 10, 10, 10, 10, 10, 10, 10, 10, 10, ...",1,"[KERNDTLB, KERNSTOR]"
2,2,"[13, 12, 13, 13, 14, 14, 15, 15, 16, 12, 12, 1...",1,[KERNSTOR]
3,3,"[40, 28, 28, 28, 29, 29, 29, 29, 29, 29, 41, 4...",1,"[APPCHILD, APPREAD]"
4,4,"[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",0,[]


In [6]:
all_causes = sorted({c for row in labeled['causes'] for c in row})
print("Distinct fault categories present across windows:", all_causes)

Distinct fault categories present across windows: ['APPCHILD', 'APPOUT', 'APPREAD', 'APPRES', 'APPSEV', 'APPTO', 'KERNDTLB', 'KERNMNTF', 'KERNREC', 'KERNRTSP', 'KERNSTOR', 'KERNTERM']
